<a href="https://colab.research.google.com/github/sxsaa/Mathematical-Reasoning-in-Small-Language-Models-using-Process-Reward-Models-and-Best-of-N-Search/blob/main/Testing_different_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl wandb

In [ ]:
# 01. Testing one math sample, purposingly giving a wrong step

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Load your NEWLY balanced model from Hugging Face!
model_id = "sxsaa/Qwen2.5-3B-Math-Verifier-FullData-v2.0"

print(f"Downloading and loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load in 16-bit precision automatically based on your GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
)
model.eval()

# 2. Define the scoring function
def evaluate_math_step(problem, history, candidate_step):
    prompt = (
        f"Problem: {problem}\n\n"
        f"History of Previous Steps:\n{history}\n\n"
        f"Current Step to Evaluate: {candidate_step}\n\n"
        f"Question: Label this step as Correct, Neutral, or Incorrect."
    )

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]

    correct_id = tokenizer.encode("Correct", add_special_tokens=False)[0]
    incorrect_id = tokenizer.encode("Incorrect", add_special_tokens=False)[0]
    neutral_id = tokenizer.encode("Neutral", add_special_tokens=False)[0]

    target_logits = logits[0, [correct_id, neutral_id, incorrect_id]]
    probs = F.softmax(target_logits, dim=-1)

    labels = ["Correct", "Neutral", "Incorrect"]
    best_idx = torch.argmax(probs).item()

    return labels[best_idx], {
        "Correct": probs[0].item(),
        "Neutral": probs[1].item(),
        "Incorrect": probs[2].item()
    }

# 3. The Test Case (Checking for complexity bias and math logic)
problem_text = "Solve the system of equations for y: 2x + y = 10 and x - y = 2."

test_cases = [
    {
        "step_num": 1,
        "history": "No previous steps.",
        "step": "First, let's isolate x in the second equation: x = y + 2.",
        "expected": "Correct"
    },
    {
        "step_num": 2,
        "history": "Step 1: First, let's isolate x in the second equation: x = y + 2.",
        "step": "Now substitute this into the first equation: 2(y + 2) + y = 10, which simplifies to 2y + 2 + y = 10.",
        "expected": "Incorrect"
    }
]

# 4. Run the Evaluation
print(f"\n--- Testing Verifier Model (v2 Balanced Data) ---\nProblem: {problem_text}\n")

for test in test_cases:
    print(f"Checking Step {test['step_num']}: '{test['step']}'")
    prediction, scores = evaluate_math_step(problem_text, test['history'], test['step'])

    match = "✅ PASS" if prediction == test['expected'] else "❌ FAIL"
    print(f"Model Prediction: **{prediction}** | Expected: {test['expected']} {match}")
    print(f"Confidence: Correct ({scores['Correct']:.1%}), Neutral ({scores['Neutral']:.1%}), Incorrect ({scores['Incorrect']:.1%})")
    print("-" * 50)

In [ ]:
# 02. Testing the "sxsaa/Qwen2.5-3B-Math-Verifier-FullData" model with 500 samples in the test data set

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm # This gives us a nice progress bar!

# 1. Load your trained model and tokenizer
model_id = "sxsaa/Qwen2.5-3B-Math-Verifier-FullData"
print(f"Loading model {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

# 2. Get token IDs for the labels
correct_id = tokenizer.encode("Correct", add_special_tokens=False)[0]
incorrect_id = tokenizer.encode("Incorrect", add_special_tokens=False)[0]
neutral_id = tokenizer.encode("Neutral", add_special_tokens=False)[0]

# Keep a map to translate the argmax index back to a string
labels_list = ["Correct", "Neutral", "Incorrect"]
token_ids = [correct_id, neutral_id, incorrect_id]

# 3. Load the test dataset you just pushed
print("Loading Test Dataset...")
# Notice we specify split="test" to pull the new data
test_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="test")

# Note: If the test set is massive and you want a quick preview first,
# uncomment the line below to test on just the first 500 samples:
test_dataset = test_dataset.select(range(500))

# 4. Set up tracking metrics
correct_predictions = 0
total_predictions = len(test_dataset)

# Track exactly which categories are succeeding or failing
label_stats = {
    "Correct": {"total": 0, "correct": 0},
    "Neutral": {"total": 0, "correct": 0},
    "Incorrect": {"total": 0, "correct": 0}
}

print(f"Starting evaluation on {total_predictions} samples...")

# 5. Evaluation Loop
for row in tqdm(test_dataset):
    # Extract the prompt and true label from the messages list
    user_prompt = row['messages'][0]['content']
    true_label = row['messages'][1]['content']

    # Track totals for the specific label
    if true_label in label_stats:
        label_stats[true_label]["total"] += 1

    # Format the prompt exactly as it was during training
    chat = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Perform inference
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :]

    # Extract the logits for our 3 specific target words
    target_logits = logits[0, token_ids]

    # Find which word had the highest probability
    best_idx = torch.argmax(target_logits).item()
    predicted_label = labels_list[best_idx]

    # Check if the model got it right
    if predicted_label == true_label:
        correct_predictions += 1
        if true_label in label_stats:
            label_stats[true_label]["correct"] += 1

# 6. Calculate and print final results
accuracy = (correct_predictions / total_predictions) * 100

print("\n" + "="*50)
print("📊 FINAL EVALUATION RESULTS 📊")
print("="*50)
print(f"Total Samples Tested: {total_predictions}")
print(f"Overall Model Accuracy: {accuracy:.2f}%\n")

print("--- Accuracy Breakdown by Label ---")
for label, stats in label_stats.items():
    if stats["total"] > 0:
        pct = (stats["correct"] / stats["total"]) * 100
        print(f"{label}: {pct:.2f}% ({stats['correct']}/{stats['total']})")
print("="*50)

In [ ]:
# 03. Testing the "sxsaa/Qwen2.5-3B-Math-Verifier-FullData-v2.0" model with 500 samples in the test data set

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm # This gives us a nice progress bar!

# 1. Load your trained model and tokenizer
model_id = "sxsaa/Qwen2.5-3B-Math-Verifier-FullData-v2.0"
print(f"Loading model {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

# 2. Get token IDs for the labels
correct_id = tokenizer.encode("Correct", add_special_tokens=False)[0]
incorrect_id = tokenizer.encode("Incorrect", add_special_tokens=False)[0]
neutral_id = tokenizer.encode("Neutral", add_special_tokens=False)[0]

# Keep a map to translate the argmax index back to a string
labels_list = ["Correct", "Neutral", "Incorrect"]
token_ids = [correct_id, neutral_id, incorrect_id]

# 3. Load the test dataset you just pushed
print("Loading Test Dataset...")
# Notice we specify split="test" to pull the new data
test_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="test")

# Note: If the test set is massive and you want a quick preview first,
# uncomment the line below to test on just the first 500 samples:
test_dataset = test_dataset.select(range(500))

# 4. Set up tracking metrics
correct_predictions = 0
total_predictions = len(test_dataset)

# Track exactly which categories are succeeding or failing
label_stats = {
    "Correct": {"total": 0, "correct": 0},
    "Neutral": {"total": 0, "correct": 0},
    "Incorrect": {"total": 0, "correct": 0}
}

print(f"Starting evaluation on {total_predictions} samples...")

# 5. Evaluation Loop
for row in tqdm(test_dataset):
    # Extract the prompt and true label from the messages list
    user_prompt = row['messages'][0]['content']
    true_label = row['messages'][1]['content']

    # Track totals for the specific label
    if true_label in label_stats:
        label_stats[true_label]["total"] += 1

    # Format the prompt exactly as it was during training
    chat = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Perform inference
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :]

    # Extract the logits for our 3 specific target words
    target_logits = logits[0, token_ids]

    # Find which word had the highest probability
    best_idx = torch.argmax(target_logits).item()
    predicted_label = labels_list[best_idx]

    # Check if the model got it right
    if predicted_label == true_label:
        correct_predictions += 1
        if true_label in label_stats:
            label_stats[true_label]["correct"] += 1

# 6. Calculate and print final results
accuracy = (correct_predictions / total_predictions) * 100

print("\n" + "="*50)
print("📊 FINAL EVALUATION RESULTS 📊")
print("="*50)
print(f"Total Samples Tested: {total_predictions}")
print(f"Overall Model Accuracy: {accuracy:.2f}%\n")

print("--- Accuracy Breakdown by Label ---")
for label, stats in label_stats.items():
    if stats["total"] > 0:
        pct = (stats["correct"] / stats["total"]) * 100
        print(f"{label}: {pct:.2f}% ({stats['correct']}/{stats['total']})")
print("="*50)

In [ ]:
# 04. Testing the "Qwen/Qwen2.5-3B" model with 500 samples in the test data set

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# 1. Load the UNTRAINED Base Model
model_id = "Qwen/Qwen2.5-3B"
print(f"Loading BASE model {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Base models sometimes need their pad token set manually
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

# 2. Get token IDs for the labels
correct_id = tokenizer.encode("Correct", add_special_tokens=False)[0]
incorrect_id = tokenizer.encode("Incorrect", add_special_tokens=False)[0]
neutral_id = tokenizer.encode("Neutral", add_special_tokens=False)[0]

labels_list = ["Correct", "Neutral", "Incorrect"]
token_ids = [correct_id, neutral_id, incorrect_id]

# 3. Load the EXACT same test dataset
print("Loading Test Dataset...")
test_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="test")

# Slice the first 500 samples to match your previous test perfectly
test_dataset = test_dataset.select(range(500))

# 4. Set up tracking metrics
correct_predictions = 0
total_predictions = len(test_dataset)

label_stats = {
    "Correct": {"total": 0, "correct": 0},
    "Neutral": {"total": 0, "correct": 0},
    "Incorrect": {"total": 0, "correct": 0}
}

print(f"Starting BASELINE evaluation on {total_predictions} samples...")

# 5. Evaluation Loop
for row in tqdm(test_dataset):
    user_prompt = row['messages'][0]['content']
    true_label = row['messages'][1]['content']

    if true_label in label_stats:
        label_stats[true_label]["total"] += 1

    chat = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :]

    target_logits = logits[0, token_ids]

    best_idx = torch.argmax(target_logits).item()
    predicted_label = labels_list[best_idx]

    if predicted_label == true_label:
        correct_predictions += 1
        if true_label in label_stats:
            label_stats[true_label]["correct"] += 1

# 6. Final Results
accuracy = (correct_predictions / total_predictions) * 100

print("\n" + "="*50)
print("📊 BASELINE MODEL RESULTS (UNTRAINED) 📊")
print("="*50)
print(f"Total Samples Tested: {total_predictions}")
print(f"Overall Model Accuracy: {accuracy:.2f}%\n")

print("--- Accuracy Breakdown by Label ---")
for label, stats in label_stats.items():
    if stats["total"] > 0:
        pct = (stats["correct"] / stats["total"]) * 100
        print(f"{label}: {pct:.2f}% ({stats['correct']}/{stats['total']})")
print("="*50)

In [ ]:
# 05. Testing the "Qwen/Qwen2.5-Math-1.5B" model with 500 samples in the test data set

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# 1. Load the UNTRAINED Base Model
model_id = "Qwen/Qwen2.5-Math-1.5B"
print(f"Loading BASE model {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Base models sometimes need their pad token set manually
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

# 2. Get token IDs for the labels
correct_id = tokenizer.encode("Correct", add_special_tokens=False)[0]
incorrect_id = tokenizer.encode("Incorrect", add_special_tokens=False)[0]
neutral_id = tokenizer.encode("Neutral", add_special_tokens=False)[0]

labels_list = ["Correct", "Neutral", "Incorrect"]
token_ids = [correct_id, neutral_id, incorrect_id]

# 3. Load the EXACT same test dataset
print("Loading Test Dataset...")
test_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="test")

# Slice the first 500 samples to match your previous test perfectly
test_dataset = test_dataset.select(range(500))

# 4. Set up tracking metrics
correct_predictions = 0
total_predictions = len(test_dataset)

label_stats = {
    "Correct": {"total": 0, "correct": 0},
    "Neutral": {"total": 0, "correct": 0},
    "Incorrect": {"total": 0, "correct": 0}
}

print(f"Starting BASELINE evaluation on {total_predictions} samples...")

# 5. Evaluation Loop
for row in tqdm(test_dataset):
    user_prompt = row['messages'][0]['content']
    true_label = row['messages'][1]['content']

    if true_label in label_stats:
        label_stats[true_label]["total"] += 1

    chat = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :]

    target_logits = logits[0, token_ids]

    best_idx = torch.argmax(target_logits).item()
    predicted_label = labels_list[best_idx]

    if predicted_label == true_label:
        correct_predictions += 1
        if true_label in label_stats:
            label_stats[true_label]["correct"] += 1

# 6. Final Results
accuracy = (correct_predictions / total_predictions) * 100

print("\n" + "="*50)
print("📊 BASELINE MODEL RESULTS (UNTRAINED) 📊")
print("="*50)
print(f"Total Samples Tested: {total_predictions}")
print(f"Overall Model Accuracy: {accuracy:.2f}%\n")

print("--- Accuracy Breakdown by Label ---")
for label, stats in label_stats.items():
    if stats["total"] > 0:
        pct = (stats["correct"] / stats["total"]) * 100
        print(f"{label}: {pct:.2f}% ({stats['correct']}/{stats['total']})")
print("="*50)

In [ ]:
# 06. Testing the "Qwen/Qwen2.5-Math-7B" model with 500 samples in the test data set

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

# 1. Load the UNTRAINED Base Model
model_id = "Qwen/Qwen2.5-Math-7B"
print(f"Loading BASE model {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Base models sometimes need their pad token set manually
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

# 2. Get token IDs for the labels
correct_id = tokenizer.encode("Correct", add_special_tokens=False)[0]
incorrect_id = tokenizer.encode("Incorrect", add_special_tokens=False)[0]
neutral_id = tokenizer.encode("Neutral", add_special_tokens=False)[0]

labels_list = ["Correct", "Neutral", "Incorrect"]
token_ids = [correct_id, neutral_id, incorrect_id]

# 3. Load the EXACT same test dataset
print("Loading Test Dataset...")
test_dataset = load_dataset("sxsaa/PRM800K-Step-Verifier", split="test")

# Slice the first 500 samples to match your previous test perfectly
test_dataset = test_dataset.select(range(500))

# 4. Set up tracking metrics
correct_predictions = 0
total_predictions = len(test_dataset)

label_stats = {
    "Correct": {"total": 0, "correct": 0},
    "Neutral": {"total": 0, "correct": 0},
    "Incorrect": {"total": 0, "correct": 0}
}

print(f"Starting BASELINE evaluation on {total_predictions} samples...")

# 5. Evaluation Loop
for row in tqdm(test_dataset):
    user_prompt = row['messages'][0]['content']
    true_label = row['messages'][1]['content']

    if true_label in label_stats:
        label_stats[true_label]["total"] += 1

    chat = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :]

    target_logits = logits[0, token_ids]

    best_idx = torch.argmax(target_logits).item()
    predicted_label = labels_list[best_idx]

    if predicted_label == true_label:
        correct_predictions += 1
        if true_label in label_stats:
            label_stats[true_label]["correct"] += 1

# 6. Final Results
accuracy = (correct_predictions / total_predictions) * 100

print("\n" + "="*50)
print("📊 BASELINE MODEL RESULTS (UNTRAINED) 📊")
print("="*50)
print(f"Total Samples Tested: {total_predictions}")
print(f"Overall Model Accuracy: {accuracy:.2f}%\n")

print("--- Accuracy Breakdown by Label ---")
for label, stats in label_stats.items():
    if stats["total"] > 0:
        pct = (stats["correct"] / stats["total"]) * 100
        print(f"{label}: {pct:.2f}% ({stats['correct']}/{stats['total']})")
print("="*50)